In [ ]:
import sys

import pandas as pd


spec = pd.read_csv("atmospheric_spectroscopy.csv")

sys = pd.read_csv("planetary_systems.csv", comment="#", low_memory=False)


sys_default = sys[sys["default_flag"] == 1].copy()

sys_default = sys_default.drop_duplicates(subset="pl_name")

merged = spec.merge(
    sys_default,
    left_on="PL_NAME",
    right_on="pl_name",
    how="left",          
    suffixes=("_spec", "_sys"),
)

print(f"Spectroscopy rows:      {len(spec)}")
print(f"Unique planets (spec):  {spec['PL_NAME'].nunique()}")
print(f"Planetary systems rows: {len(sys_default)}")
print(f"Merged rows:            {len(merged)}")
unmatched = merged[merged["pl_name"].isna()]["PL_NAME"].unique()
print(f"Unmatched planet names: {list(unmatched)}")


merged.to_csv("merged_dataset.csv", index=False)
print("Saved merged_dataset.csv")

In [ ]:


import math
import sys
import pandas as pd



EARTH_RADIUS = 1.0          
EARTH_DENSITY = 5.51       
EARTH_ESC_VEL = 11.19       
EARTH_TEMP = 288.0         

WEIGHTS = {
    "radius": 0.57,
    "density": 1.07,
    "esc_vel": 0.70,
    "temp": 5.58,
}
N_PROPERTIES = 4


def esi_component(value, reference, weight):
    if pd.isna(value) or value <= 0:
        return None
    ratio_term = abs(value - reference) / (value + reference)
    return (1 - ratio_term) ** (weight / N_PROPERTIES)


def compute_escape_velocity_ratio(mass_earth, radius_earth):
    """Escape velocity relative to Earth: v_esc / v_esc_earth = sqrt(M/R) in Earth units."""
    if pd.isna(mass_earth) or pd.isna(radius_earth) or radius_earth <= 0:
        return None
    return math.sqrt(mass_earth / radius_earth) * EARTH_ESC_VEL


def compute_esi(row):
    radius = row.get("pl_rade")
    density = row.get("pl_dens")
    mass = row.get("pl_masse")
    temp = row.get("pl_eqt")

    esc_vel = compute_escape_velocity_ratio(mass, radius)

    esi_radius = esi_component(radius, EARTH_RADIUS, WEIGHTS["radius"])
    esi_density = esi_component(density, EARTH_DENSITY, WEIGHTS["density"])
    esi_escvel = esi_component(esc_vel, EARTH_ESC_VEL, WEIGHTS["esc_vel"])
    esi_temp = esi_component(temp, EARTH_TEMP, WEIGHTS["temp"])

    interior_parts = [x for x in (esi_radius, esi_density) if x is not None]
    surface_parts = [x for x in (esi_escvel, esi_temp) if x is not None]

    if not interior_parts or not surface_parts:
        return None

    esi_interior = math.prod(interior_parts) ** (1 / len(interior_parts))
    esi_surface = math.prod(surface_parts) ** (1 / len(surface_parts))

    return math.sqrt(esi_interior * esi_surface)


def esi_to_category(esi):
    if esi is None:
        return "Insufficient Data"
    if esi >= 0.8:
        return "Earth-like (High Habitability Potential)"
    elif esi >= 0.6:
        return "Moderately Habitable"
    elif esi >= 0.4:
        return "Marginally Habitable"
    else:
        return "Non-Habitable"


def classify_planet_type(radius, mass):
    """Standard radius/mass-based exoplanet classification bins used across exoplanet science."""
    if pd.isna(radius):
        return "Unknown"
    if radius < 1.5:
        return "Rocky"
    elif radius < 2.0:
        return "Super-Earth"
    elif radius < 4.0:
        return "Sub-Neptune"
    elif radius < 10.0:
        return "Neptune-like"
    else:
        return "Gas Giant"


def infer_atmosphere(planet_type, temp):
    
    if planet_type == "Unknown":
        return "Unknown (insufficient data)"

    if planet_type == "Gas Giant":
        return "Inferred: H/He-dominated primordial envelope"

    if planet_type == "Neptune-like":
        return "Inferred: H/He + volatile ices (Neptune-like envelope)"

    if planet_type == "Sub-Neptune":
        if pd.notna(temp) and temp > 800:
            return "Inferred: H/He envelope likely eroding (highly irradiated)"
        return "Inferred: possible H/He or steam/volatile-rich envelope"

    # Rocky or Super-Earth
    if pd.isna(temp):
        return "Inferred: thin secondary atmosphere (temperature unknown)"
    if temp > 1000:
        return "Inferred: atmosphere likely stripped / exotic vapor (extreme heat)"
    elif temp > 500:
        return "Inferred: thin or no atmosphere (too hot for stable volatiles)"
    elif 180 <= temp <= 320:
        return "Inferred: possible CO2/N2/H2O secondary atmosphere (temperate)"
    else:
        return "Inferred: thin atmosphere or frozen volatiles (cold)"

import sys
import pandas as pd
import numpy as np

def main():
    if len(sys.argv) != 3:
        print("Usage: python habitability_v2.py <input_csv> <output_csv>")
        sys.exit(1)

    input_path = sys.argv[1]
    output_path = sys.argv[2]

    df = pd.read_csv(input_path)

    # Compute Earth Similarity Index (ESI)
    
    esi_values: pd.Series = df.apply(compute_esi, axis=1)

    df["esi"] = esi_values
    df["habitability_category"] = esi_values.apply(esi_to_category)

    df["planet_type"] = df.apply(
        lambda row: classify_planet_type(
            row.get("pl_rade"),
            row.get("pl_masse"),
        ),
        axis=1,
    )

    df["inferred_atmosphere"] = df.apply(
        lambda row: infer_atmosphere(
            row["planet_type"],
            row.get("pl_eqt"),
        ),
        axis=1,
    )

    df.to_csv(output_path, index=False)

    print("\nHabitability category counts:")
    print(df["habitability_category"].value_counts(dropna=False).to_string())

    print("\nPlanet type counts:")
    print(df["planet_type"].value_counts(dropna=False).to_string())

    print("\nTop 10 by ESI:")

    top = (
        df.dropna(subset=["esi"])
          .sort_values("esi", ascending=False)
          .drop_duplicates(subset="pl_name")
          .head(10)
    )

    print(
        top[
            [
                "pl_name",
                "esi",
                "habitability_category",
                "planet_type",
                "inferred_atmosphere",
            ]
        ].to_string(index=False)
    )

    print(f"\nSaved to: {output_path}")


if __name__ == "__main__":
    main()

In [ ]:
df = pd.read_csv("exoplanet_details.csv")

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
print(df.dtypes)

In [ ]:
print(df.describe(include="all"))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("exoplanet_details.csv")

numeric_df = df.select_dtypes(include="number")

plt.figure(figsize=(16, 12))
sns.heatmap(
    numeric_df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("exoplanet_details.csv")


features = ["pl_rade", "pl_eqt", "st_teff"]


numeric_df = df.select_dtypes(include="number")

correlations = numeric_df.corr()[features]

print(correlations)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv("exoplanet_details.csv")


features = ["pl_rade", "pl_eqt", "st_teff"]


numeric_df = df.select_dtypes(include="number")


corr = numeric_df.corr()[features]


corr = corr.drop(index=features, errors="ignore")


corr["mean_abs"] = corr.abs().mean(axis=1)
corr = corr.sort_values("mean_abs", ascending=False)
corr = corr.drop(columns="mean_abs")

# Plot
plt.figure(figsize=(10, max(12, len(corr) * 0.25)))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.3
)

plt.title("Correlation of All Numerical Features with Selected Features")
plt.xlabel("Selected Features")
plt.ylabel("Other Features")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv("exoplanet_details.csv")


targets = ["pl_rade", "pl_eqt", "st_teff"]

numeric_df = df.select_dtypes(include="number")


corr = numeric_df.corr()

target_corr = corr[targets].drop(index=targets, errors="ignore")


abs_corr = target_corr.abs()


max_corr = abs_corr.max(axis=1)

mean_corr = abs_corr.mean(axis=1)


results = pd.DataFrame({
    "Max_Abs_Correlation": max_corr,
    "Mean_Abs_Correlation": mean_corr
})

top_10 = results.sort_values(
    "Max_Abs_Correlation"
).head(10)

print("\nTop 10 features least correlated with all 3 targets:\n")
print(top_10)


plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_10.reset_index(),
    x="Max_Abs_Correlation",
    y="index"
)

plt.xlabel("Maximum Absolute Correlation")
plt.ylabel("Feature")
plt.title("Top 10 Features Least Correlated with the 3 Selected Features")

plt.tight_layout()
plt.show()

In [ ]:

features_to_delete = top_10.index.tolist()

print("Deleting:")
print(features_to_delete)

df = df.drop(columns=features_to_delete)

print("\nNew shape:", df.shape)

In [ ]:

missing_percent = df.isnull().mean() * 100


columns_to_delete = missing_percent[missing_percent > 60].index.tolist()

print("Columns with more than 60% missing values:")
print(columns_to_delete)

print("\nNumber of columns deleted:", len(columns_to_delete))


df = df.drop(columns=columns_to_delete)

print("\nNew dataset shape:", df.shape)

In [ ]:
df


In [ ]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from typing import Tuple, List, cast
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)




CSV_PATH = r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\Exoplanet_Habitable\final_df.csv"

MISSING_THRESHOLD = 0.60

# Number of useful features retained.
# Increase this if you want the models to have more information.
TOP_FEATURES = 25

RANDOM_STATE = 42


TARGETS = [
    "habitability_category",
    "planet_type",
    "inferred_atmosphere"
]



def load_data(csv_path: str) -> pd.DataFrame:

    df = pd.read_csv(csv_path)

    print("Dataset Loaded Successfully")

    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    print("\nColumn Names:\n")

    for col in df.columns:
        print(col)

    print("\nFirst 5 Rows:\n")
    print(df.head())

    return df




def remove_metadata(df: pd.DataFrame):

    df = df.copy()

    # Exact columns that identify a planet, paper, observation,
    # coordinate, reference, or database record.
    identifier_columns = [

        # Planet / system identity
        "PL_NAME",
        "pl_name",
        "hostname",
        "pl_letter",

        # Database identifiers
        "rowid",
        "tic_id",
        "gaia_id",
        "gaia_dr2_id",
        "gaia_dr3_id",

        # References / publications
        "AUTHORS",
        "disc_refname",
        "pl_refname",
        "st_refname",
        "sy_refname",
        "disc_pubdate",
        "pl_pubdate",
        "releasedate",
        "rowupdate",

        # Discovery metadata
        "disc_facility",
        "disc_telescope",
        "disc_instrument",
        "disc_locale",

        # Observation metadata
        "INSTRUMENT",
        "FACILITY",
        "NOTE",

        # Coordinates
        "rastr",
        "decstr",
        "dec",
        "glat",
        "glon",
        "elat",

        # Miscellaneous identifiers / solution information
        "solution_id",
        "default_flag"
    ]

    pattern_columns = []

    for col in df.columns:

        col_lower = col.lower()

        if (
            col_lower.endswith("_name")
            or col_lower.endswith("_id")
            or "refname" in col_lower
        ):
            pattern_columns.append(col)

    columns_to_remove = set(identifier_columns + pattern_columns)


    columns_to_remove -= set(TARGETS)

    existing = [
        c for c in columns_to_remove
        if c in df.columns
    ]

    df.drop(columns=existing, inplace=True)

    print("\nMetadata / identifier removal")
    print(f"Dropped {len(existing)} columns.")

    print("\nRemoved columns:")

    for col in existing:
        print(" -", col)

    return df




def remove_high_missing(df: pd.DataFrame):

    df = df.copy()

    missing_ratio = df.isna().mean()

    high_missing = missing_ratio[
        missing_ratio > MISSING_THRESHOLD
    ].index.tolist()

    # Never remove targets
    high_missing = [
        c for c in high_missing
        if c not in TARGETS
    ]

    df.drop(
        columns=high_missing,
        inplace=True
    )

    print("\nHigh-missing feature removal")
    print(
        f"Dropped {len(high_missing)} columns "
        f"with > {MISSING_THRESHOLD * 100:.0f}% missing values."
    )

    if high_missing:

        print("\nRemoved:")

        for col in high_missing:
            print(" -", col)

    return df




def preprocess_data(df: pd.DataFrame):

    df = df.copy()


    y = {}

    target_encoders = {}

    for target in TARGETS:

        encoder = LabelEncoder()

        y[target] = encoder.fit_transform(
            df[target].astype(str)
        )

        target_encoders[target] = encoder

 

    X = df.drop(
        columns=TARGETS
    ).copy()



    numeric_columns = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    numeric_imputer = SimpleImputer(
        strategy="median"
    )

    if numeric_columns:

        X[numeric_columns] = numeric_imputer.fit_transform(
            X[numeric_columns]
        )



    categorical_columns = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    if categorical_columns:

        for col in categorical_columns:

            X[col] = (
                X[col]
                .fillna("Unknown")
                .astype(str)
            )

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    if categorical_columns:

        X[categorical_columns] = (
            categorical_encoder.fit_transform(
                X[categorical_columns]
            )
        )

    print("\nPreprocessing complete.")

    print(
        f"Numeric features     : {len(numeric_columns)}"
    )

    print(
        f"Categorical features : {len(categorical_columns)}"
    )

    return (
        X,
        y,
        numeric_columns,
        categorical_columns,
        numeric_imputer,
        categorical_encoder,
        target_encoders
    )




def select_important_features(X, y):


    print("FEATURE SELECTION")
  

    importance_tables = []


    for target in TARGETS:

        print(
            f"\nCalculating feature importance for: {target}"
        )

        model = RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"
        )

        model.fit(
            X,
            y[target]
        )

        importance = pd.DataFrame({
            "Feature": X.columns,
            "Importance": model.feature_importances_
        })

        importance["Target"] = target

        importance_tables.append(
            importance
        )

    importance_all = pd.concat(
        importance_tables,
        ignore_index=True
    )


    combined_importance = (
        importance_all
        .groupby("Feature")["Importance"]
        .mean()
        .sort_values(ascending=False)
    )

    selected_features = (
        combined_importance
        .head(TOP_FEATURES)
        .index
        .tolist()
    )

    print("\n" + "=" * 60)
    print(f"TOP {TOP_FEATURES} SELECTED FEATURES")
    print("=" * 60)

    for i, feature in enumerate(
        selected_features,
        start=1
    ):

        print(
            f"{i:2d}. {feature:<35} "
            f"{combined_importance[feature]:.6f}"
        )

    return (
        selected_features,
        combined_importance,
        importance_all
    )




def train_models(X, y, selected_features):

    X_selected = X[
        selected_features
    ].copy()

    print("\nTraining using:")
    print(f"{len(selected_features)} selected features")


    model_habitability = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    )

    model_habitability.fit(
        X_selected,
        y["habitability_category"]
    )


    model_planet_type = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        n_jobs=-1
    )

    model_planet_type.fit(
        X_selected,
        y["planet_type"]
    )


    model_atmosphere = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        n_jobs=-1
    )

    model_atmosphere.fit(
        X_selected,
        y["inferred_atmosphere"]
    )

    return (
        model_habitability,
        model_planet_type,
        model_atmosphere
    )



def evaluate_model(
    model,
    X,
    y,
    target,
    target_encoder
):

    predictions = model.predict(X)

    accuracy = accuracy_score(
        y[target],
        predictions
    )


    print(target.upper())


    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print("\nClassification Report")

    labels = np.arange(len(target_encoder.classes_))

    print(
        classification_report(
            y[target],
            predictions,
            labels=labels,
            target_names=target_encoder.classes_,
            zero_division=0
        )
    )

    print("\nConfusion Matrix")

    print(
        confusion_matrix(
            y[target],
            predictions
        )
    )




def save_models(
    model_habitability,
    model_planet_type,
    model_atmosphere,
    numeric_imputer,
    categorical_encoder,
    target_encoders,
    selected_features,
    numeric_columns,
    categorical_columns
):

    joblib.dump(
        model_habitability,
        "habitability_model.pkl"
    )

    joblib.dump(
        model_planet_type,
        "planet_type_model.pkl"
    )

    joblib.dump(
        model_atmosphere,
        "atmosphere_model.pkl"
    )

    joblib.dump(
        numeric_imputer,
        "numeric_imputer.pkl"
    )

    joblib.dump(
        categorical_encoder,
        "ordinal_encoder.pkl"
    )

    joblib.dump(
        target_encoders,
        "target_encoders.pkl"
    )

    # IMPORTANT:
    # Save exactly which features the models expect.
    joblib.dump(
        selected_features,
        "selected_features.pkl"
    )

    joblib.dump(
        numeric_columns,
        "numeric_columns.pkl"
    )

    joblib.dump(
        categorical_columns,
        "categorical_columns.pkl"
    )

    print("\nModels and preprocessing objects saved successfully.")



def predict_new_planet():


    print("NEW PLANET PREDICTION")
  



    habitability_model = joblib.load(
        "habitability_model.pkl"
    )

    planet_type_model = joblib.load(
        "planet_type_model.pkl"
    )

    atmosphere_model = joblib.load(
        "atmosphere_model.pkl"
    )

    numeric_imputer = joblib.load(
        "numeric_imputer.pkl"
    )

    categorical_encoder = joblib.load(
        "ordinal_encoder.pkl"
    )

    target_encoders = joblib.load(
        "target_encoders.pkl"
    )

    selected_features = joblib.load(
        "selected_features.pkl"
    )

    numeric_columns = joblib.load(
        "numeric_columns.pkl"
    )

    categorical_columns = joblib.load(
        "categorical_columns.pkl"
    )



    print(
        "\nThe model determined that these "
        "features are useful for prediction:\n"
    )

    for i, column in enumerate(
        selected_features,
        start=1
    ):

        print(
            f"{i:2d}. {column}"
        )

    print(
        "\nEnter values below."
    )

    print(
        "Press ENTER if a value is unknown.\n"
    )

    new_data = {}



    for column in selected_features:

        if column in numeric_columns:

            while True:

                value = input(
                    f"{column}: "
                ).strip()

                if value == "":
                    new_data[column] = np.nan
                    break

                try:

                    new_data[column] = float(
                        value
                    )

                    break

                except ValueError:

                    print(
                        "Please enter a numeric value."
                    )

        else:

            value = input(
                f"{column}: "
            ).strip()

            if value == "":
                value = "Unknown"

            new_data[column] = value


    new_df = pd.DataFrame(
        [new_data]
    )



    numeric_selected = [
        c for c in selected_features
        if c in numeric_columns
    ]

    if numeric_selected:

        new_df[numeric_selected] = (
            numeric_imputer.transform(
                new_df[numeric_selected]
            )
        )



    categorical_selected = [
        c for c in selected_features
        if c in categorical_columns
    ]

    if categorical_selected:

        for col in categorical_selected:

            new_df[col] = (
                new_df[col]
                .fillna("Unknown")
                .astype(str)
            )

        new_df[categorical_selected] = (
            categorical_encoder.transform(
                new_df[categorical_selected]
            )
        )



    new_df = new_df[
        selected_features
    ]



    habitability_prediction = (
        habitability_model.predict(
            new_df
        )[0]
    )

    planet_prediction = (
        planet_type_model.predict(
            new_df
        )[0]
    )

    atmosphere_prediction = (
        atmosphere_model.predict(
            new_df
        )[0]
    )


    habitability_label = (
        target_encoders[
            "habitability_category"
        ].inverse_transform(
            [habitability_prediction]
        )[0]
    )

    planet_label = (
        target_encoders[
            "planet_type"
        ].inverse_transform(
            [planet_prediction]
        )[0]
    )

    atmosphere_label = (
        target_encoders[
            "inferred_atmosphere"
        ].inverse_transform(
            [atmosphere_prediction]
        )[0]
    )




    print("PREDICTION RESULT")


    print(
        "\nHabitability       :",
        habitability_label
    )

    print(
        "Planet Type        :",
        planet_label
    )

    print(
        "Inferred Atmosphere:",
        atmosphere_label
    )

    return (
        habitability_label,
        planet_label,
        atmosphere_label
    )


df = load_data(
    CSV_PATH
)



df = remove_metadata(df)


df = remove_high_missing(df)

print(
    "\nShape after cleaning:",
    df.shape
)

(
    X,
    y,
    numeric_columns,
    categorical_columns,
    numeric_imputer,
    categorical_encoder,
    target_encoders
) = preprocess_data(df)



SelectionResult = Tuple[
    List[str],
    pd.Series,
    pd.DataFrame
]

selection_result = cast(
    SelectionResult,
    select_important_features(X, y)
)

(
    selected_features,
    combined_importance,
    importance_all
) = selection_result

# Reduce X to selected features
X_selected = X[
    selected_features
].copy()

print(
    "\nFinal model input shape:",
    X_selected.shape
)



(
    model_habitability,
    model_planet_type,
    model_atmosphere
) = train_models(
    X,
    y,
    selected_features
)



evaluate_model(
    model_habitability,
    X_selected,
    y,
    "habitability_category",
    target_encoders[
        "habitability_category"
    ]
)

evaluate_model(
    model_planet_type,
    X_selected,
    y,
    "planet_type",
    target_encoders[
        "planet_type"
    ]
)

evaluate_model(
    model_atmosphere,
    X_selected,
    y,
    "inferred_atmosphere",
    target_encoders[
        "inferred_atmosphere"
    ]
)




print("FINAL FEATURE IMPORTANCE")

print(
    combined_importance
    .head(TOP_FEATURES)
    .to_string()
)



save_models(
    model_habitability,
    model_planet_type,
    model_atmosphere,
    numeric_imputer,
    categorical_encoder,
    target_encoders,
    selected_features,
    numeric_columns,
    categorical_columns
)



choice = input(
    "\nPredict a new planet? (y/n): "
).strip().lower()

if choice == "y":

    predictions = predict_new_planet()

else:

    print(
        "\nPrediction skipped."
    )

In [ ]:
import base64

encoded_string = "bWFkZSBieSB2cnVzaGFiaC5zIHN3cw=="
print(base64.b64decode(encoded_string).decode('utf-8'))
